# Vector Databases with Pinecone — a hands-on tutorial

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

```
 Part 1 — one small example                    Part 2 — a real knowledge base
 ──────────────────────────                    ──────────────────────────────
 "total_amount is the money paid…"             kb_feed_ecom.xlsx (6 tabs, 345 rows)
        │ OpenAI embedding                             │ openpyxl: read each tab
        ▼                                              ▼
 [0.012, -0.034, … 1536 numbers]               one record per row: id + numbers + metadata
        │ upsert                                       │ embed 100 at a time, upsert
        ▼                                              ▼
 ┌──────────── Pinecone index "ecom-kb-tutorial" (in the cloud) ──────────────┐
 │  namespace "first-example"  (5 records)    namespace "ecom-kb"  (337)     │
 └────────────────────────────────────────────────────────────────────────────┘
        ▲                                              ▲
   query: "Which parcels arrived late?"        query · filter · find a similar SQL example
```

| Idea | In one line |
|---|---|
| Embedding | Text turned into a list of numbers that captures its **meaning** |
| Similarity | Texts with similar meaning have numbers that are close |
| Vector database | Stores the numbers + labels (metadata), and finds the closest ones fast |
| Index | Pinecone's container for vectors — fixed size (1536) and fixed way of comparing (cosine) |
| Namespace | A separate section inside one index — searches never mix namespaces |
| Metadata filter | "Closest — but only from this tab / this table" |

## 0. Setup

**Accounts:** an OpenAI API key, and a free Pinecone account (app.pinecone.io → **API Keys** → create a key).

Run these **in the terminal** (PyCharm Terminal), not in the notebook:

| Step | Mac | Windows (PowerShell) |
|---|---|---|
| 1. Go to the folder | `cd ~/Downloads/vector_db_tutorial` | `cd ~\Downloads\vector_db_tutorial` |
| 2. Create a venv | `python3 -m venv venv` | `python -m venv venv` |
| 3. Activate it | `source venv/bin/activate` | `venv\Scripts\activate` |
| 4. Install packages | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| 5. Add your keys | `cp .env.example .env`, then edit `.env` | `Copy-Item .env.example .env`, then edit `.env` |

`.env` must contain:

```
OPENAI_API_KEY=sk-...
PINECONE_API_KEY=pcsk_...
```

Cost: embedding everything here uses about 60,000 tokens with `text-embedding-3-small` — a fraction of a cent.
The Pinecone index fits in the free plan.

In [ ]:
import json
import math
import os
import time

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv()                                        # reads both keys from .env

client = OpenAI()                                    # finds OPENAI_API_KEY by itself
EMBED_MODEL = "text-embedding-3-small"               # 1536 numbers per text
DIMENSION = 1536

INDEX_NAME = "ecom-kb-tutorial"                      # lower-case letters, digits and "-" only

print("OpenAI key  :", os.getenv("OPENAI_API_KEY", "")[:7] + "…")
print("Pinecone key:", os.getenv("PINECONE_API_KEY", "")[:7] + "…")

---
# Part 1 — one small example

## 1. What is an embedding?

```
 "total_amount is the money paid for an order"
        │
        ▼   OpenAI embedding model
 [0.0123, -0.0441, 0.0087, … ]   ← 1536 numbers
```

The numbers mean nothing on their own. What matters: **texts with similar meaning get similar numbers.**
We write one small function and use it everywhere in this notebook.

In [ ]:
def embed(texts):
    # Send a list of texts to OpenAI, get back one list of 1536 numbers per text.
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


vector = embed(["total_amount is the money paid for an order"])[0]

print("How many numbers:", len(vector))
print("First five      :", [round(x, 4) for x in vector[:5]])

## 2. Close in meaning = close in numbers

To compare two embeddings we use **cosine similarity**: 1.0 means the same direction (same meaning),
values near 0 mean unrelated. Plain Python, no libraries:

```
            a · b                 a · b  = sum of a[i] × b[i]
 cos = ─────────────────         |a|    = square root of (sum of a[i]²)
         |a|  ×  |b|
```

In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(y * y for y in b))
    return dot / (length_a * length_b)


question = "How much revenue did we make?"
sentences = [
    "total_amount is the money paid for an order",
    "stock_quantity is how many units sit in a warehouse",
    "rating is the 1 to 5 stars a customer gave",
]

question_vector = embed([question])[0]
sentence_vectors = embed(sentences)

print(f"Question: {question}\n")
for sentence, vec in zip(sentences, sentence_vectors):
    print(f"{cosine_similarity(question_vector, vec):.3f}  {sentence}")

Which sentence scores highest? The question shares **no** words with the money sentence ("revenue" vs "money paid for an order"),
yet it should come out on top. That is **semantic search**: matching by meaning, not by shared words.
A keyword search for "revenue" would find nothing here.

Comparing one question with every stored text, one by one, is fine for 3 sentences — not for 3 million.
A **vector database** stores the vectors and finds the closest ones fast.

## 3. Connect to Pinecone and create an index

```
 Pinecone (your account, in the cloud)
   └── index "ecom-kb-tutorial"          dimension 1536 · metric cosine · AWS us-east-1
         ├── namespace "first-example"   ← Part 1
         └── namespace "ecom-kb"         ← Part 2
               each record:  id · values (1536 numbers) · metadata (labels + the text)
```

Two settings are fixed when the index is created and can't be changed later:

| Setting | Value | Why |
|---|---|---|
| `dimension` | 1536 | Must equal the length of every vector — the size `text-embedding-3-small` produces |
| `metric` | `cosine` | How "close" is measured — the formula from section 2 |

`ServerlessSpec(cloud="aws", region="us-east-1")` is where Pinecone runs it; the free plan uses this region.

In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

if not pc.has_index(INDEX_NAME):                    # create it only the first time
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Created index", INDEX_NAME)
else:
    print("Index already exists:", INDEX_NAME)

index = pc.Index(INDEX_NAME)                        # the handle we read and write with
print(index.describe_index_stats())

## 4. Insert (upsert): id + values + metadata

Each record is a dict with three keys:

| Key | Holds |
|---|---|
| `id` | a unique text, e.g. `"sales_1"` |
| `values` | the 1536 numbers from `embed()` |
| `metadata` | a dict of labels to filter on — **and the text itself**, so we can read it back |

Pinecone stores the numbers, not your sentence. If you want the text back in search results, put it in the metadata.

`upsert` = insert, or replace if the id already exists — running the cell twice is safe.

In [ ]:
FIRST = "first-example"                              # the namespace for Part 1

texts = [
    "total_amount is the money paid for an order, before shipping and tax.",
    "delivery_date is the day the parcel reached the customer.",
    "stock_quantity is how many units of a product sit in a warehouse.",
    "rating is the 1 to 5 stars a customer gave a product.",
    "payment_status tells if a payment completed, failed or was refunded.",
]
ids = ["sales_1", "delivery_1", "inventory_1", "reviews_1", "payments_1"]
topics = ["sales", "delivery", "inventory", "reviews", "payments"]

vectors = embed(texts)

records = []
for record_id, text, topic, values in zip(ids, texts, topics, vectors):
    records.append({"id": record_id, "values": values, "metadata": {"topic": topic, "text": text}})

index.upsert(vectors=records, namespace=FIRST)
print("Sent", len(records), "records")

Pinecone takes a few seconds before new records show up in counts and searches. This helper waits until they do.

In [ ]:
def wait_for(namespace, expected, seconds=60):
    for _ in range(seconds // 2):
        stats = index.describe_index_stats()
        found = stats.namespaces.get(namespace)
        count = found.vector_count if found else 0
        if count >= expected:
            print(f"{namespace}: {count} records ready")
            return
        time.sleep(2)
    print(f"{namespace}: still {count} after {seconds} s — run this cell again")


wait_for(FIRST, 5)

## 5. Retrieve: embed the question, ask for the closest records

```
 "Which parcels arrived late?" ──embed──► [1536 numbers] ──query──► top_k closest records
```

Pinecone returns a **score**. With `metric="cosine"` the score **is** the cosine similarity from section 2:
**higher = closer**.

In [ ]:
question = "Which parcels arrived late?"

result = index.query(
    vector=embed([question])[0],
    top_k=2,
    namespace=FIRST,
    include_metadata=True,                           # give back the labels and the text
)

print(f"Question: {question}\n")
for match in result.matches:
    print(f"{match.score:.3f}  {match.id:12}  {match.metadata['text']}")

## 6. Filter with metadata

"Find the closest — but only among payment records." The `filter` runs on the metadata.

In [ ]:
result = index.query(
    vector=embed(["money we gave back to customers"])[0],
    top_k=2,
    namespace=FIRST,
    filter={"topic": {"$eq": "payments"}},
    include_metadata=True,
)
for match in result.matches:
    print(f"{match.score:.3f}  {match.id:12}  {match.metadata['text']}")

**Recap:** text → `embed()` → numbers. `upsert` stores id + numbers + metadata (with the text inside).
`query` embeds the question the same way and returns the closest records with a score. `filter` narrows the search.

**Next:** keep the index — **Notebook 02** loads a whole spreadsheet into it.

**Try it yourself**

1. Upsert one record of your own into `first-example` and find it with a question that shares **no** words with it.
2. Change `top_k` to 5 in section 5. Are the extra results useful?
3. Filter on a topic that doesn't exist (`"$eq": "marketing"`). What comes back?